In [13]:
#vou gerar uma tabela um pouco aleatória para treinar dados
import sqlite3
import pandas as pd
import numpy as np

# 1. Conexão com o banco de dados SQLite local
conexao = sqlite3.connect('banco_credito_massivo.db')
ponteiro = conexao.cursor()

# 2. Criação da tabela de análise de risco
ponteiro.execute('''
    CREATE TABLE IF NOT EXISTS analise_credito (
        id_cliente INTEGER PRIMARY KEY,
        nome_cliente TEXT,
        renda_anual REAL,
        valor_parcela REAL
    )
''')

# 3. Gerador de dados massivos de alta fidelidade (10.000 clientes)
np.random.seed(42)
nomes_base = ['Carlos Silva', 'Ana Santos', 'Bruno Souza', 'Diana Lima', 'Eduardo Costa', 'Fabiana Oliveira']

dados_massivos = []
for i in range(1, 10001):
    # Simulando ruídos de strings (espaços extras nas pontas e letras maiúsculas/minúsculas zoadas)
    nome_cru = np.random.choice(nomes_base)
    if i % 3 == 0:
        nome_cru = f"   {nome_cru}   "
    if i % 5 == 0:
        nome_cru = nome_cru.lower()
    elif i % 7 == 0:
        nome_cru = nome_cru.upper()
        
    # Geração dos dados numéricos (Renda anual entre 30k e 150k, Parcelas entre 500 e 5000)
    renda = float(np.random.randint(30000, 150000))
    parcela = float(np.random.randint(500, 5000))
    
    dados_massivos.append((i, nome_cru, renda, parcela))

# 4. Injeta os dados na tabela e salva
ponteiro.execute("DELETE FROM analise_credito") # Garante tabela limpa para reexecuções
ponteiro.executemany('INSERT INTO analise_credito VALUES (?, ?, ?, ?)', dados_massivos)
conexao.commit()

print("📊 Banco SQLite gerado")


📊 Banco SQLite gerado


In [14]:
query = """
SELECT *
FROM analise_credito
"""
table_cred = pd.read_sql_query(query,  conexao)
table_cred.describe()
#coloquei esse passo para extarir os dados e conhecer um pouco a tabela gerada

,id_cliente,renda_anual,valor_parcela
count,10000.00000,10000.000000,10000.000000
mean,5000.50000,89447.040700,2731.508100
std,2886.89568,34672.561993,1295.301452
min,1.00000,30005.000000,500.000000
25%,2500.75000,58958.500000,1604.750000
50%,5000.50000,89319.000000,2733.000000
75%,7500.25000,119256.500000,3849.000000
max,10000.00000,149991.000000,4999.000000


In [16]:
table_cred.head(10)
#conhecendo um pouco mais

,id_cliente,nome_cliente,renda_anual,valor_parcela
0,1,Diana Lima,30860.0,4272.0
1,2,Eduardo Costa,84886.0,966.0
2,3,Bruno Souza,117498.0,3944.0
3,4,Diana Lima,90263.0,630.0
4,5,fabiana oliveira,94820.0,1269.0
5,6,Diana Lima,94925.0,2933.0
6,7,DIANA LIMA,134724.0,1684.0
7,8,Diana Lima,115305.0,4617.0
8,9,Eduardo Costa,123016.0,974.0
9,10,bruno souza,114478.0,2547.0


In [17]:
frame_cred = pd.DataFrame(table_cred)#Passando para dataframe
print (frame_cred.isna().sum())#procurando dados faltantes
print(frame_cred.apply(lambda col: col.astype(str).str.match(r'^\s*$')).sum().sum())#procurando espaços em branco também

id_cliente       0
nome_cliente     0
renda_anual      0
valor_parcela    0
dtype: int64
0


In [18]:
frame_cred["comprometimento_renda"] = frame_cred['valor_parcela']/(frame_cred['renda_anual']/12)*100
comprometido = frame_cred['risco'] = frame_cred['comprometimento_renda'] >30
print(comprometido.sum())

6115


In [19]:
tabela_risco = pd.concat([frame_cred['nome_cliente'], frame_cred['renda_anual'], frame_cred['comprometimento_renda'], frame_cred['risco']], axis = 1)
tabela_risco2 = tabela_risco[tabela_risco['risco']]
tabela_risco2.describe()

,renda_anual,comprometimento_renda
count,6115.000000,6115.000000
mean,78297.643500,60.881286
std,32445.196154,29.525788
min,30005.000000,30.023938
25%,50746.000000,39.406460
50%,73237.000000,50.766337
75%,103743.000000,73.621197
max,149989.000000,186.429061


In [28]:
# EXECUTE ESTE BLOCO APENAS PARA CRIAR A SEGUNDA TABELA NO BANCO
ponteiro.execute('''
    CREATE TABLE IF NOT EXISTS historico_pagamentos (
        id_cliente INTEGER PRIMARY KEY,
        status_atraso TEXT
    )
''')
# Injeta dados simulados: clientes ímpares têm atraso, pares estão em dia
dados_pagamentos = [(i, 'Com Atraso' if i % 2 != 0 else 'Em Dia') for i in range(1, 10001)]
ponteiro.execute("DELETE FROM historico_pagamentos")
ponteiro.executemany('INSERT INTO historico_pagamentos VALUES (?, ?)', dados_pagamentos)
conexao.commit()



In [21]:
analise_credito2 = pd.concat([frame_cred['nome_cliente'], frame_cred['renda_anual']], axis = 1)
analise_credito2.head()

,nome_cliente,renda_anual
0,Diana Lima,30860.0
1,Eduardo Costa,84886.0
2,Bruno Souza,117498.0
3,Diana Lima,90263.0
4,fabiana oliveira,94820.0


In [25]:
table_cred.to_sql(
    name = 'analise_credito2',
    con = conexao,
    if_exists = 'replace',
    index = False,
)

10000

In [55]:
query = """
SELECT a.nome_cliente, a.renda_anual, b.id_cliente, b.status_atraso
    FROM analise_credito2 a
    LEFT JOIN historico_pagamentos b
    ON a.id_cliente = b.id_cliente
"""

dataf = pd.read_sql_query(query, conexao)
dataf.head(10)

,nome_cliente,renda_anual,id_cliente,status_atraso
0,Diana Lima,30860.0,1,Com Atraso
1,Eduardo Costa,84886.0,2,Em Dia
2,Bruno Souza,117498.0,3,Com Atraso
3,Diana Lima,90263.0,4,Em Dia
4,fabiana oliveira,94820.0,5,Com Atraso
5,Diana Lima,94925.0,6,Em Dia
6,DIANA LIMA,134724.0,7,Com Atraso
7,Diana Lima,115305.0,8,Em Dia
8,Eduardo Costa,123016.0,9,Com Atraso
9,bruno souza,114478.0,10,Em Dia
